# KINETIX Studio v2 — Colab GPU Worker Bridge

Runs ComfyUI and `kinetix_bridge.py` on a Google Colab GPU instance (A100 / L4 / T4), links cached models from Google Drive (`MyDrive/kinetix_models/`), copies the bridge and workflow files from `MyDrive/kinetix_deploy/`, exposes port `8000` over a Cloudflare Quick Tunnel, and prints a ready-to-open `?gpu=` Studio link.

In [ ]:
# 1. Settings & Bridge Token
import secrets

SITE_URL = "https://your-kinetix-site.vercel.app"
BRIDGE_TOKEN = ""

if not BRIDGE_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        BRIDGE_TOKEN = (userdata.get("KINETIX_BRIDGE_TOKEN") or "").strip()
    except Exception:
        BRIDGE_TOKEN = ""

if not BRIDGE_TOKEN:
    BRIDGE_TOKEN = secrets.token_urlsafe(24)
    print(f"[KINETIX] Generated BRIDGE_TOKEN: {BRIDGE_TOKEN}")
    print("[KINETIX] NOTE: Paste this token into the Studio GPU pill, or set KINETIX_GPU_URL (tunnel URL) + KINETIX_GPU_TOKEN (this value) on Vercel.")
else:
    print("[KINETIX] Loaded BRIDGE_TOKEN from configuration / Colab userdata.")


In [ ]:
# 2. Detect GPU & select VRAM_FLAG
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    gpu_name = torch.cuda.get_device_name(0)
else:
    vram_gb = 0.0
    gpu_name = "No CUDA GPU"

VRAM_FLAG = "--highvram" if vram_gb >= 31.0 else "--normalvram"
print(f"[KINETIX] GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) -> {VRAM_FLAG}")


In [ ]:
# 3. Mount Drive, clone ComfyUI, install dependencies & symlink models
import os
from pathlib import Path
import subprocess
from google.colab import drive  # type: ignore

drive.mount("/content/drive")

DRIVE_MODELS_DIR = Path("/content/drive/MyDrive/kinetix_models")
DRIVE_DEPLOY_DIR = Path("/content/drive/MyDrive/kinetix_deploy")
COMFY_DIR = Path("/content/ComfyUI")

DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

if not COMFY_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)],
        check=True,
    )

subprocess.run(
    [
        "pip",
        "install",
        "-r",
        str(COMFY_DIR / "requirements.txt"),
        "fastapi",
        "uvicorn",
        "requests",
        "websocket-client",
        "python-multipart",
        "pydantic",
    ],
    check=True,
)

MODEL_SUBFOLDERS = [
    "diffusion_models",
    "text_encoders",
    "vae",
    "checkpoints",
    "loras",
    "ultralytics",
    "sams",
]

linked_count = 0
for sub in MODEL_SUBFOLDERS:
    src_sub = DRIVE_MODELS_DIR / sub
    dst_sub = COMFY_DIR / "models" / sub
    src_sub.mkdir(parents=True, exist_ok=True)
    dst_sub.mkdir(parents=True, exist_ok=True)
    for item in src_sub.iterdir():
        if item.is_file():
            target = dst_sub / item.name
            if not target.exists() and not target.is_symlink():
                os.symlink(str(item), str(target))
                linked_count += 1

print(f"[KINETIX] ComfyUI ready at {COMFY_DIR} ({linked_count} model files symlinked).")


In [ ]:
# 4. Copy kinetix_bridge.py and workflows/ from MyDrive/kinetix_deploy/ to /content/
from pathlib import Path
import shutil

DRIVE_DEPLOY_DIR = Path("/content/drive/MyDrive/kinetix_deploy")
CONTENT_DIR = Path("/content")

src_bridge = DRIVE_DEPLOY_DIR / "kinetix_bridge.py"
if not src_bridge.is_file():
    raise FileNotFoundError(
        f"Missing {src_bridge}. Please upload the contents of the repository 'deploy/' folder "
        f"(including kinetix_bridge.py and workflows/) to {DRIVE_DEPLOY_DIR} in Google Drive."
    )

shutil.copy2(src_bridge, CONTENT_DIR / "kinetix_bridge.py")

src_workflows = DRIVE_DEPLOY_DIR / "workflows"
dst_workflows = CONTENT_DIR / "workflows"
dst_workflows.mkdir(parents=True, exist_ok=True)
if src_workflows.is_dir():
    shutil.copytree(src_workflows, dst_workflows, dirs_exist_ok=True)

print(f"[KINETIX] Copied kinetix_bridge.py and workflows/ to {CONTENT_DIR}.")


In [ ]:
# 5. Start ComfyUI (port 8188) and Kinetix Bridge (port 8000)
import os
import subprocess
import time
import requests

comfy_log = open("/content/comfyui.log", "a", encoding="utf-8")
comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "127.0.0.1", "--port", "8188", VRAM_FLAG],
    cwd="/content/ComfyUI",
    stdout=comfy_log,
    stderr=subprocess.STDOUT,
)

print("[KINETIX] Waiting up to 180s for ComfyUI on port 8188...")
comfy_ready = False
deadline = time.time() + 180
while time.time() < deadline:
    try:
        r = requests.get("http://127.0.0.1:8188/queue", timeout=3)
        if r.status_code == 200:
            comfy_ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not comfy_ready:
    raise RuntimeError("ComfyUI did not become ready within 180s. Check /content/comfyui.log.")
print("[KINETIX] ComfyUI is online on port 8188.")

bridge_env = os.environ.copy()
bridge_env["COMFY_DIR"] = "/content/ComfyUI"
bridge_env["COMFY_URL"] = "http://127.0.0.1:8188"
bridge_env["KINETIX_BRIDGE_TOKEN"] = BRIDGE_TOKEN
bridge_env["WORKFLOW_PATH"] = "/content/workflows/kinetix_pro.api.json"

bridge_log = open("/content/bridge.log", "a", encoding="utf-8")
bridge_proc = subprocess.Popen(
    ["uvicorn", "kinetix_bridge:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content",
    env=bridge_env,
    stdout=bridge_log,
    stderr=subprocess.STDOUT,
)

print("[KINETIX] Waiting for Kinetix Bridge /health on port 8000...")
bridge_ready = False
deadline = time.time() + 60
while time.time() < deadline:
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.status_code == 200:
            bridge_ready = True
            print("[KINETIX] Bridge health:", r.json())
            break
    except Exception:
        pass
    time.sleep(1)

if not bridge_ready:
    raise RuntimeError("Kinetix Bridge failed to start on port 8000. Check /content/bridge.log.")


In [ ]:
# 6. Start Cloudflare Quick Tunnel & print ?gpu= Share Link
from pathlib import Path
import re
import stat
import subprocess
import time
import requests

cf_bin = Path("/content/cloudflared")
if not cf_bin.exists():
    cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    with requests.get(cf_url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with cf_bin.open("wb") as f:
            for chunk in resp.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

cf_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 60

while time.time() < deadline:
    line = cf_proc.stdout.readline() if cf_proc.stdout else ""
    if line:
        match = url_pattern.search(line)
        if match:
            tunnel_url = match.group(0)
            break
    elif cf_proc.poll() is not None:
        break
    else:
        time.sleep(0.2)

if not tunnel_url:
    raise RuntimeError("Timed out waiting 60s for Cloudflare Quick Tunnel URL.")

share_link = f"{SITE_URL.rstrip('/')}/studio?gpu={tunnel_url}"
print("=" * 72)
print(f"[KINETIX] Tunnel URL : {tunnel_url}")
print(f"[KINETIX] Share Link : {share_link}")
print("=" * 72)


In [ ]:
# 7. Keep-alive loop (prints /health every 60s)
import time
import requests

print("[KINETIX] Keep-alive monitor started (Ctrl+C / Stop cell to exit)...")
try:
    while True:
        try:
            res = requests.get("http://127.0.0.1:8000/health", timeout=5)
            print(time.strftime("%Y-%m-%d %H:%M:%S"), res.json())
        except Exception as exc:
            print(time.strftime("%Y-%m-%d %H:%M:%S"), f"Health check error: {exc}")
        time.sleep(60)
except KeyboardInterrupt:
    print("[KINETIX] Keep-alive monitor stopped.")
